# **5. Hypothesis Testing & P-Values**

## What is Hypothesis Testing?
A statistical method to make decisions about population parameters using sample data.

## The Two Hypotheses

| Hypothesis | Symbol | Description |
|------------|--------|-------------|
| **Null Hypothesis** | H₀ | No effect, no difference (status quo) |
| **Alternative Hypothesis** | H₁ or Hₐ | There IS an effect or difference |

## The Process
1. State H₀ and H₁
2. Choose significance level (α), usually 0.05
3. Collect data and compute test statistic
4. Calculate p-value
5. Make decision: Reject H₀ if p-value < α

---
# **Understanding P-Values**

## Definition
The **p-value** is the probability of observing data as extreme as (or more extreme than) what we observed, **assuming the null hypothesis is true**.

$$\text{P-value} = P(\text{data} | H_0 \text{ is true})$$

## Interpretation
| P-value | Interpretation |
|---------|---------------|
| p < 0.01 | Strong evidence against H₀ |
| p < 0.05 | Moderate evidence against H₀ |
| p < 0.10 | Weak evidence against H₀ |
| p ≥ 0.10 | Little to no evidence against H₀ |

## Common Misconceptions ⚠️
- p-value is NOT P(H₀ is true)
- p-value is NOT the probability of making an error
- Statistically significant ≠ practically important

In [ ]:
# ===============================================
# ONE-SAMPLE T-TEST
# ===============================================

import numpy as np
from scipy import stats

print("=" * 50)
print("ONE-SAMPLE T-TEST")
print("=" * 50)

# Scenario: A company claims their widget weighs 50g on average.
# We sample 25 widgets. Does our data support their claim?

np.random.seed(42)
# Sample data (actually comes from population with mean=52)
sample = np.random.normal(52, 5, 25)

claimed_mean = 50
sample_mean = np.mean(sample)
sample_std = np.std(sample, ddof=1)
n = len(sample)

print(f"\nClaim: μ = {claimed_mean}g")
print(f"\nSample statistics:")
print(f"  n = {n}")
print(f"  x̄ = {sample_mean:.2f}g")
print(f"  s = {sample_std:.2f}g")

# Hypotheses:
print("\n" + "-" * 50)
print("Hypotheses:")
print("  H₀: μ = 50 (claim is correct)")
print("  H₁: μ ≠ 50 (two-tailed test)")

# Perform t-test
t_stat, p_value = stats.ttest_1samp(sample, claimed_mean)

print(f"\nTest Results:")
print(f"  t-statistic = {t_stat:.3f}")
print(f"  p-value = {p_value:.4f}")

# Decision at α = 0.05
alpha = 0.05
print(f"\n" + "=" * 50)
if p_value < alpha:
    print(f"✗ REJECT H₀ (p = {p_value:.4f} < α = {alpha})")
    print(f"  Evidence suggests the true mean is NOT {claimed_mean}g")
else:
    print(f"✓ FAIL TO REJECT H₀ (p = {p_value:.4f} ≥ α = {alpha})")
    print(f"  Not enough evidence to reject the claim")

In [ ]:
# ===============================================
# TWO-SAMPLE T-TEST (A/B Testing)
# ===============================================

import numpy as np
from scipy import stats

print("=" * 50)
print("TWO-SAMPLE T-TEST (A/B Testing)")
print("=" * 50)

# Scenario: Testing two website versions
# Does Version B lead to more time on page than Version A?

np.random.seed(42)
# Time on page (seconds)
version_A = np.random.normal(45, 15, 100)  # Control
version_B = np.random.normal(52, 18, 100)  # Treatment

print(f"\nVersion A (Control): n={len(version_A)}, mean={np.mean(version_A):.1f}s")
print(f"Version B (Treatment): n={len(version_B)}, mean={np.mean(version_B):.1f}s")
print(f"Difference: {np.mean(version_B) - np.mean(version_A):.1f}s")

print("\n" + "-" * 50)
print("Hypotheses:")
print("  H₀: μ_B = μ_A (no difference)")
print("  H₁: μ_B > μ_A (Version B is better)")

# Independent samples t-test
t_stat, p_value_two_tailed = stats.ttest_ind(version_B, version_A)
p_value_one_tailed = p_value_two_tailed / 2  # One-tailed for "greater than"

print(f"\nTest Results (one-tailed):")
print(f"  t-statistic = {t_stat:.3f}")
print(f"  p-value = {p_value_one_tailed:.4f}")

alpha = 0.05
print(f"\n" + "=" * 50)
if t_stat > 0 and p_value_one_tailed < alpha:
    print(f"✓ SIGNIFICANT! Version B performs better.")
    print(f"  (p = {p_value_one_tailed:.4f} < α = {alpha})")
else:
    print(f"✗ NOT SIGNIFICANT. No clear winner.")

In [ ]:
# ===============================================
# VISUALIZING P-VALUES
# ===============================================

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Using our one-sample t-test example
t_stat = 2.5  # Example t-statistic
df = 24       # degrees of freedom

x = np.linspace(-4, 4, 200)
y = stats.t.pdf(x, df)

fig, ax = plt.subplots(figsize=(10, 5))

# Plot t-distribution
ax.plot(x, y, 'b-', linewidth=2, label='t-distribution (df=24)')

# Shade p-value regions (two-tailed)
x_right = np.linspace(t_stat, 4, 100)
x_left = np.linspace(-4, -t_stat, 100)

ax.fill_between(x_right, stats.t.pdf(x_right, df), alpha=0.4, color='red')
ax.fill_between(x_left, stats.t.pdf(x_left, df), alpha=0.4, color='red')

# Mark the test statistic
ax.axvline(x=t_stat, color='red', linestyle='--', linewidth=2)
ax.axvline(x=-t_stat, color='red', linestyle='--', linewidth=2)

# P-value
p_value = 2 * (1 - stats.t.cdf(t_stat, df))

ax.text(t_stat + 0.2, 0.15, f't = {t_stat}', fontsize=12, color='red')
ax.text(0, 0.2, f'P-value = {p_value:.4f}\n(shaded area)', 
        fontsize=12, ha='center', bbox=dict(boxstyle='round', facecolor='wheat'))

ax.set_xlabel('t-statistic', fontsize=12)
ax.set_ylabel('Probability Density', fontsize=12)
ax.set_title('P-Value Visualization (Two-Tailed Test)', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
# **Type I and Type II Errors**

| | H₀ True | H₀ False |
|---|---------|----------|
| **Reject H₀** | Type I Error (α) | ✓ Correct |
| **Fail to Reject H₀** | ✓ Correct | Type II Error (β) |

## Definitions
- **Type I Error (α)**: False positive - rejecting H₀ when it's true
- **Type II Error (β)**: False negative - failing to reject H₀ when it's false
- **Power (1-β)**: Probability of correctly rejecting a false H₀

In [ ]:
# ===============================================
# TYPE I AND TYPE II ERRORS SIMULATION
# ===============================================

import numpy as np
from scipy import stats

np.random.seed(42)

print("=" * 50)
print("TYPE I AND TYPE II ERRORS")
print("=" * 50)

n_simulations = 1000
sample_size = 30
alpha = 0.05

# TYPE I ERROR: H₀ is TRUE, but we reject it
print("\n--- Type I Error (False Positive) ---")
print("Reality: μ = 100 (H₀ is TRUE)")

type1_errors = 0
for _ in range(n_simulations):
    sample = np.random.normal(100, 15, sample_size)  # True mean IS 100
    _, p = stats.ttest_1samp(sample, 100)
    if p < alpha:
        type1_errors += 1

print(f"Rejected H₀: {type1_errors}/{n_simulations} = {type1_errors/n_simulations:.1%}")
print(f"Expected (α): {alpha:.1%}")

# TYPE II ERROR: H₀ is FALSE, but we fail to reject
print("\n--- Type II Error (False Negative) ---")
print("Reality: μ = 105 (H₀ is FALSE, true difference exists)")

type2_errors = 0
for _ in range(n_simulations):
    sample = np.random.normal(105, 15, sample_size)  # True mean is NOT 100
    _, p = stats.ttest_1samp(sample, 100)  # Testing against H₀: μ=100
    if p >= alpha:
        type2_errors += 1

print(f"Failed to reject H₀: {type2_errors}/{n_simulations} = {type2_errors/n_simulations:.1%}")
print(f"Power (1-β): {1 - type2_errors/n_simulations:.1%}")

---
## Summary

| Concept | Key Point |
|---------|----------|
| **H₀** | Null hypothesis (no effect) |
| **H₁** | Alternative hypothesis (there IS an effect) |
| **P-value** | P(data \| H₀ true) |
| **α** | Significance level (usually 0.05) |
| **Reject H₀** | When p-value < α |
| **Type I Error** | False positive (reject true H₀) |
| **Type II Error** | False negative (fail to reject false H₀) |

### Python Reference
```python
from scipy import stats

# One-sample t-test
stats.ttest_1samp(sample, mu_0)

# Two-sample t-test
stats.ttest_ind(sample1, sample2)
```